# Module 04 — Lazy Evaluation, Actions & Reuse

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

spark = begin_module('module-04-lazy-execution')
ui_links()

In [ ]:
from pyspark.sql import functions as F, Window
trips = spark.read.parquet(*config.yellow_taxi_paths())
expensive = (
    trips.filter('total_amount > 10')
    .withColumn('rn', F.row_number().over(Window.partitionBy('VendorID').orderBy(F.desc('total_amount'))))
    .filter('rn <= 100')
    .groupBy('PULocationID')
    .agg(F.avg('total_amount').alias('avg_fare'), F.count('*').alias('n'))
)
expensive.explain('formatted')

## Each action can trigger a new job

In [ ]:
_, t1 = timed_action(expensive)
_, t2 = timed_action(expensive, action='collect')
print(f'count={t1:.2f}s, collect={t2:.2f}s — check UI job list')

## Branching DAG — duplicate work

In [ ]:
branch_a = expensive.filter('n > 50')
branch_b = expensive.filter('avg_fare > 30')
_, ta = timed_action(branch_a)
_, tb = timed_action(branch_b)
print(f'without cache: a={ta:.2f}s b={tb:.2f}s (shared ancestor recomputed)')

## Spark Observation (metrics without double full scan for logging)

In [ ]:
from pyspark.sql import Observation
obs = Observation('avg_rows')
observed = trips.filter('total_amount > 50').observe(obs, F.count(F.lit(1)).alias('rows'))
observed.limit(1).collect()  # minimal action to run pipeline
print('observed rows metric:', obs.get('rows'))

## cache / persist

In [ ]:
expensive.cache()
_, c1 = timed_action(expensive)
_, c2 = timed_action(expensive.filter('n > 10'))
expensive.unpersist()
print(f'with cache: first={c1:.2f}s second={c2:.2f}s')

## checkpoint — cuts lineage (serialization cost)

In [ ]:
ckpt_dir = str(config.CHECKPOINT_DIR)
spark.sparkContext.setCheckpointDir(ckpt_dir)
ck = expensive.checkpoint(eager=True)
ck.explain('formatted')
_, tc = timed_action(ck)
print(f'checkpointed action: {tc:.2f}s')

## Intermediate Iceberg table vs cache

In [ ]:
drop_iceberg_table(spark, 'local.default.lazy_intermediate')
expensive.writeTo('local.default.lazy_intermediate').using('iceberg').create()
from_table = spark.table('local.default.lazy_intermediate')
from_table.explain('formatted')  # file scan + stats, fresh optimizer view
_, tt = timed_action(from_table)
print(f'read table: {tt:.2f}s')

## Takeaways

1. Lazy until action; branching without reuse is expensive.
2. Observation for metrics; cache/checkpoint for reuse; table for stats + longevity.